# Sprint 4: Reentrenamiento Fase 2 — LR ajustado

**Diagnóstico del entrenamiento anterior:**
- Fase 1 (CNN congelada): AUC_macro = 0.8326 en 15 epochs ✓
- Fase 2 (full FT): solo corrió 12 epochs antes de early stopping (+0.002)
- Causa: LR_vit=1e-5 demasiado bajo → el modelo no aprende nada nuevo

**Ajustes:**
- LR_cnn: 1e-6 → 2e-6  (levemente más alto)
- LR_vit: 1e-5 → 5e-5  (5× más alto, permite aprendizaje real)
- Scheduler: ReduceLROnPlateau → CosineAnnealingLR (más estable para FT)
- Epochs: 30 → 40
- Patience: 8 → 15

**Punto de partida:** checkpoint de Fase 1 (`sprint4_phase1_best.pt`).
No repetimos Fase 1 — ya convergió bien.

**Runtime: T4 GPU. Tiempo estimado: ~60-90 min.**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR     = f'{PROJECT_ROOT}/code'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

import os, sys, shutil, time
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ CPU'}")

H5_LOCAL     = '/content/nih_images.h5'
PROCESSED_S4 = '/content/processed_s4'
CLASS_NAMES  = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}


In [ ]:
!pip install -q torchxrayvision scipy h5py
print('deps OK')


## 2. Verificar HDF5 y CSVs del Sprint 4

In [ ]:
import h5py, pandas as pd

# HDF5
if not (os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 1_000_000_000):
    print('Copiando HDF5...')
    shutil.copy2(f'{NIH_DIR}/nih_images.h5', H5_LOCAL)
with h5py.File(H5_LOCAL, 'r') as hf:
    print(f'✓ HDF5: {len(hf["images"]):,} imágenes')

# CSVs Sprint 4
os.makedirs(PROCESSED_S4, exist_ok=True)
for csv in ['train.csv', 'val.csv', 'test.csv']:
    local = f'{PROCESSED_S4}/{csv}'
    if not os.path.exists(local):
        shutil.copy(f'{PROJECT_ROOT}/data/processed_s4/{csv}', local)
    df = pd.read_csv(local)
    print(f'✓ {csv}: {len(df)} imgs')

# Verificar que el checkpoint de Fase 1 existe
p1_ckpt = f'{PROJECT_ROOT}/experiments_s4/checkpoints/sprint4_phase1_best.pt'
if os.path.exists(p1_ckpt):
    size_mb = os.path.getsize(p1_ckpt) / (1024**2)
    print(f'✓ Checkpoint Fase 1: {size_mb:.1f} MB')
else:
    print('✗ Checkpoint Fase 1 no encontrado. Debes correr la Fase 1 primero.')


## 3. Cargar modelo desde checkpoint Fase 1

In [ ]:
# Limpiar cache
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'): del sys.modules[mod]

import torch
from src.models.cnn_vit import build_cnn_vit
from src.training.utils import load_config, get_device

cfg    = load_config(f'{CODE_DIR}/configs/sprint4.yaml')
device = get_device()

model = build_cnn_vit(cfg).to(device)

# Cargar pesos de Fase 1
p1_ckpt = f'{PROJECT_ROOT}/experiments_s4/checkpoints/sprint4_phase1_best.pt'
ckpt = torch.load(p1_ckpt, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])

print(f'✓ Checkpoint Fase 1 cargado')
print(f'  Fase: {ckpt["phase"]} | Epoch: {ckpt["epoch"]} | '
      f'Val AUC_macro: {ckpt["best_val_auc_macro"]:.4f}')

# Desbloquear todos los parámetros para Fase 2
model.set_phase(2)
summary = model.param_summary()
print(f'\nParámetros para Fase 2:')
print(f'  CNN: {summary["cnn_total"]:,}')
print(f'  ViT: {summary["vit_total"]:,}')
print(f'  Total entrenables: {summary["trainable"]:,}')


## 4. DataLoaders

In [ ]:
from torch.utils.data import DataLoader
from src.datasets.nih_hdf5 import NIHDatasetHDF5
from src.datasets.transforms import build_train_transform, build_eval_transform

train_tfm = build_train_transform(224)
eval_tfm  = build_eval_transform(224)

train_ds = NIHDatasetHDF5(f'{PROCESSED_S4}/train.csv', H5_LOCAL, train_tfm)
val_ds   = NIHDatasetHDF5(f'{PROCESSED_S4}/val.csv',   H5_LOCAL, eval_tfm)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print(f'Distribución train: {train_ds.class_distribution()}')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)


## 5. Fase 2 con LR ajustado y CosineAnnealingLR

**Cambios respecto al entrenamiento anterior:**
- LR_vit: 1e-5 → **5e-5** (permite aprendizaje real en Fase 2)
- LR_cnn: 1e-6 → **2e-6** (levemente más alto)
- Scheduler: ReduceLROnPlateau → **CosineAnnealingLR** (más estable)
- Patience: 8 → **15** (más tiempo para que el scheduler ayude)
- Epochs: **40**


In [ ]:
import copy
import numpy as np
import torch.nn as nn
from torch.amp import GradScaler, autocast
from src.evaluation.metrics_s2 import compute_metrics_multiclass, logits_to_probs_multiclass
from src.training.utils import save_checkpoint

# ─── Hiperparámetros Fase 2 ajustados ───
LR_CNN   = 2e-6   # antes: 1e-6
LR_VIT   = 5e-5   # antes: 1e-5
EPOCHS   = 40
PATIENCE = 15
T_MAX    = 40     # periodo del coseno

# ─── Criterion con class weights ────────
def compute_class_weights(loader, num_classes, device):
    counts = torch.zeros(num_classes)
    for batch in loader:
        for label in batch['label']:
            counts[label.item()] += 1
    total   = counts.sum()
    weights = total / (num_classes * counts)
    return (weights / weights.sum() * num_classes).to(device)

print('Calculando class weights...')
weights   = compute_class_weights(train_loader, 4, device)
criterion = nn.CrossEntropyLoss(weight=weights)
print(f'Class weights: {weights.tolist()}')

# ─── Optimizer con LR diferenciado ──────
param_groups = model.get_param_groups(lr_cnn=LR_CNN, lr_vit=LR_VIT)
optimizer    = torch.optim.Adam(param_groups, weight_decay=1e-4)

# ─── CosineAnnealingLR ──────────────────
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=T_MAX, eta_min=1e-7
)

scaler = GradScaler(device='cuda')

print(f'\nFase 2 ajustada:')
print(f'  LR CNN: {LR_CNN:.1e} | LR ViT: {LR_VIT:.1e}')
print(f'  Scheduler: CosineAnnealingLR (T_max={T_MAX})')
print(f'  Epochs: {EPOCHS} | Patience: {PATIENCE}')
print(f'  Entrenables: {model.trainable_params():,}')


In [ ]:
# ─── Loop de entrenamiento Fase 2 ajustada ───

CKPT_DIR = f'{PROJECT_ROOT}/experiments_s4/checkpoints'
BEST_CKPT = f'{CKPT_DIR}/sprint4_phase2v2_best.pt'

best_auc   = ckpt['best_val_auc_macro']  # AUC de fase 1 como baseline
best_state = None
no_improve = 0
history_p2v2 = []

print('=' * 70)
print(f'FASE 2v2 | Partiendo de AUC_macro={best_auc:.4f} (Fase 1)')
print('=' * 70)

for epoch in range(1, EPOCHS + 1):
    lr_cnn_now = optimizer.param_groups[0]['lr']
    lr_vit_now = optimizer.param_groups[1]['lr']

    # ── Train ──
    model.train()
    losses, all_probs, all_labels = [], [], []
    for batch in train_loader:
        imgs   = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', dtype=torch.float16):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        losses.append(loss.item())
        with torch.no_grad():
            p = logits_to_probs_multiclass(logits.detach().float())
            all_probs.append(p.cpu()); all_labels.append(labels.cpu())

    tr_metrics = compute_metrics_multiclass(
        torch.cat(all_labels).numpy(), torch.cat(all_probs).numpy()
    )

    # ── Val ──
    model.eval()
    v_losses, v_probs, v_labels = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            imgs   = batch['image'].to(device, non_blocking=True)
            labels = batch['label'].to(device, non_blocking=True)
            logits = model(imgs)
            v_losses.append(criterion(logits, labels).item())
            p = logits_to_probs_multiclass(logits.float())
            v_probs.append(p.cpu()); v_labels.append(labels.cpu())

    va_metrics = compute_metrics_multiclass(
        torch.cat(v_labels).numpy(), torch.cat(v_probs).numpy()
    )

    scheduler.step()

    print(f'[P2v2] Epoch {epoch:2d}/{EPOCHS} | '
          f'LR_cnn={lr_cnn_now:.1e} LR_vit={lr_vit_now:.1e} | '
          f'train AUC={tr_metrics["auc_macro"]:.4f} | '
          f'val AUC={va_metrics["auc_macro"]:.4f} '
          f'acc={va_metrics["accuracy"]:.4f}')
    for i, auc in enumerate(va_metrics['auc_per_class']):
        print(f'  val AUC clase {i} ({CLASS_NAMES[i]}): {auc:.4f}')

    history_p2v2.append({
        'epoch': epoch, 'lr_cnn': lr_cnn_now, 'lr_vit': lr_vit_now,
        'train_loss': float(np.mean(losses)), 'train_metrics': tr_metrics,
        'val_loss':   float(np.mean(v_losses)), 'val_metrics': va_metrics,
    })

    score = va_metrics['auc_macro']
    if score > best_auc:
        best_auc   = score
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
        torch.save({
            'phase': 'P2v2', 'epoch': epoch,
            'model_state_dict': best_state,
            'best_val_auc_macro': best_auc,
        }, BEST_CKPT)
        print(f'  → NUEVO MEJOR: AUC_macro={best_auc:.4f} ✓')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'  Early stopping (patience={PATIENCE})')
            break

print()
print(f'Fase 2v2 terminada. Mejor AUC_macro = {best_auc:.4f}')
print(f'Checkpoint: {BEST_CKPT}')


## 6. Guardar history y comparativa completa

In [ ]:
import json

# Cargar history anterior de Sprint 4
h4_path = f'{PROJECT_ROOT}/experiments_s4/history_s4.json'
with open(h4_path) as f:
    d4_old = json.load(f)

# Combinar: history original + nueva fase 2
history_combined = d4_old['history'] + [
    {**h, 'phase': 'P2v2'} for h in history_p2v2
]

result_updated = {
    **d4_old,
    'history': history_combined,
    'best_val_auc_phase2v2': best_auc,
    'checkpoint_phase2v2': BEST_CKPT,
}

h4_path_v2 = f'{PROJECT_ROOT}/experiments_s4/history_s4_v2.json'
with open(h4_path_v2, 'w') as f:
    json.dump(result_updated, f, indent=2, ensure_ascii=False)
print(f'✓ History guardado: {h4_path_v2}')


## 7. Curvas y comparativa final S2 vs S4 vs S4v2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

with open(f'{PROJECT_ROOT}/experiments_s2/history_s2.json') as f:
    d2 = json.load(f)

hist2     = d2['history']
hist4_p2  = [h for h in d4_old['history'] if h.get('phase') == 'P2']
hist4_p2v2 = history_p2v2

# Resumen numérico
best_s2   = d2['history'][d2['best_epoch']-1]['val_metrics']
best_p2   = max(hist4_p2,   key=lambda h: h['val_metrics']['auc_macro'])['val_metrics']
best_p2v2 = max(hist4_p2v2, key=lambda h: h['val_metrics']['auc_macro'])['val_metrics']

print('COMPARATIVA FINAL (val set):')
print('=' * 72)
print(f"{'Métrica':<18} {'Sprint 2':>9} {'S4 Fase2':>10} {'S4 Fase2v2':>12} {'Delta v2':>10}")
print('-' * 72)

for key, label in [
    ('auc_macro',  'AUC macro'),
    ('accuracy',   'Accuracy'),
    ('sensitivity_macro', 'Sens macro'),
    ('specificity_macro', 'Spec macro'),
]:
    v2  = best_s2.get(key, 0)
    v4  = best_p2.get(key, 0)
    v4v = best_p2v2.get(key, 0)
    delta = v4v - v2
    arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '≈')
    print(f"{label:<18} {v2:>9.4f} {v4:>10.4f} {v4v:>12.4f} {delta:>+10.4f} {arrow}")

print()
print(f"{'Clase':<15} {'S2 AUC':>8} {'S4 AUC':>8} {'S4v2 AUC':>10} {'Delta':>8}")
print('-' * 55)
for i in range(4):
    a2  = best_s2['auc_per_class'][i]
    a4  = best_p2['auc_per_class'][i]
    a4v = best_p2v2['auc_per_class'][i]
    delta = a4v - a2
    arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '≈')
    print(f"{CLASS_NAMES[i]:<15} {a2:>8.4f} {a4:>8.4f} {a4v:>10.4f} {delta:>+8.4f} {arrow}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors4 = ['#2196F3','#F44336','#4CAF50','#FF9800']

# AUC macro comparativa
ax = axes[0]
ep2   = [h['epoch'] for h in hist2]
auc2  = [h['val_metrics']['auc_macro'] for h in hist2]
ep4   = [h['epoch'] for h in hist4_p2]
auc4  = [h['val_metrics']['auc_macro'] for h in hist4_p2]
ep4v  = [h['epoch'] for h in hist4_p2v2]
auc4v = [h['val_metrics']['auc_macro'] for h in hist4_p2v2]

ax.plot(ep2,  auc2,  's-', ms=3, label=f'Sprint 2 CNN (best={max(auc2):.4f})',   color='coral')
ax.plot(ep4,  auc4,  'o--',ms=3, label=f'S4 Fase2 (best={max(auc4):.4f})',       color='steelblue', alpha=0.5)
ax.plot(ep4v, auc4v, 'o-', ms=3, label=f'S4 Fase2v2 (best={max(auc4v):.4f})',   color='steelblue')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax.set_title('Val AUC Macro: Sprint 2 vs CNN-ViT')
ax.set_xlabel('Epoch'); ax.set_ylabel('AUC macro')
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)

# AUC por clase CNN-ViT v2
ax2 = axes[1]
for i in range(4):
    aucs = [h['val_metrics']['auc_per_class'][i] for h in hist4_p2v2]
    ax2.plot(ep4v, aucs, 'o-', ms=3, label=CLASS_NAMES[i], color=colors4[i])
ax2.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax2.set_title('CNN-ViT v2: Val AUC por clase')
ax2.set_xlabel('Epoch'); ax2.legend(fontsize=8)
ax2.grid(alpha=0.3); ax2.set_ylim(0, 1.05)

best_final = max(auc4v)
plt.suptitle(f'S2={max(auc2):.4f} → S4={max(auc4):.4f} → S4v2={best_final:.4f}', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments_s4/figures/curves_s4v2.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('✓ Gráfica guardada.')


## 8. Siguiente paso

Con estos resultados:
- Si AUC macro v2 > AUC macro v1 → usar checkpoint `sprint4_phase2v2_best.pt`.
- Si sin mejora → quedarse con `sprint4_phase2_best.pt`.

Vuelve al chat para recibir:
1. Evaluación final sobre test set (Sprint 4).
2. Tabla comparativa Sprint 1 vs Sprint 2 vs Sprint 4 sobre test.
3. Grad-CAM del CNN-ViT.
4. Documentación y cierre del Sprint 4.
